### 导出 onnx

In [ ]:
import yaml
import torch

from ultralytics.nn.modules.head import Detect
from ultralytics.nn.tasks import DetectionModel


default_cfg = yaml.load(open("ultralytics/cfg/default.yaml", "r"), Loader=yaml.FullLoader)
model_cfg = yaml.load(open("cfgs/yolov12.yaml", "r"), Loader=yaml.FullLoader)
model_cfg["scale"] = "n"

model = DetectionModel(cfg=model_cfg, ch=3, verbose=False)
pth = torch.load("logs/20260525-174123/model_epoch_339_0.6336.pth", map_location="cpu")
model.load_state_dict(pth["model"])
# model.fuse(True)
model = model.eval()

for m in model.modules():
    if isinstance(m, Detect):
        m.dynamic = False  # 禁用动态grid
        m.export = True    # 启用导出模式
        m.format = 'onnx'


dummy_input = torch.randn(1, 3, 160, 160)
model_out = model(dummy_input)
print(len(model_out))
for item in model_out:
    print(item.shape)
torch.onnx.export(
    model,
    dummy_input,
    "onnx/yolov12_test.onnx",
    opset_version=12,
    do_constant_folding=True,
    input_names=["input1"],
    output_names=["output0", "output1"],
    dynamo=None,
    dynamic_axes=None,
)

# simplify ONNX model
import onnx
import onnxsim
model_onnx = onnx.load("onnx/yolov12_test.onnx")
model_onnx, check = onnxsim.simplify(model_onnx)
onnx.save(model_onnx, "onnx/yolov12_test.onnx")

from fvcore import nn as fvnn

params = fvnn.parameter_count_table(model)
print(params)

### 导出save model

In [ ]:
import yaml
import torch

from ultralytics.nn.modules.head import Detect
from ultralytics.nn.tasks import DetectionModel


default_cfg = yaml.load(open("ultralytics/cfg/default.yaml", "r"), Loader=yaml.FullLoader)
model_cfg = yaml.load(open("cfgs/yolov12.yaml", "r"), Loader=yaml.FullLoader)
model_cfg["scale"] = "n"

model = DetectionModel(cfg=model_cfg, ch=3, verbose=False)
pth = torch.load("logs/v4.1/model_epoch_359_0.6361.pth", map_location="cpu")
model.load_state_dict(pth["model"])
model = model.eval()

for m in model.modules():
    if isinstance(m, Detect):
        m.dynamic = False  # 禁用动态grid
        m.export = True    # 启用导出模式
        m.format = 'onnx'

dummy_input = torch.randn(1, 3, 160, 160)
model_out = model(dummy_input)
print(len(model_out))
for item in model_out:
    print(item.shape)
    
torch.onnx.export(
    model,
    dummy_input,
    "onnx/v4.1/yolov12_160_sm.onnx",
    opset_version=12,
    do_constant_folding=True,
    input_names=["input1"],
    output_names=["output0", "output1"],
    dynamo=None,
    dynamic_axes=None,
)

# simplify ONNX model
import onnx
import onnxslim
model_onnx = onnx.load("onnx/v4.1/yolov12_160_sm.onnx")
model_onnx = onnxslim.slim(model_onnx)
onnx.save(model_onnx, "onnx/v4.1/yolov12_160_sm.onnx")

# export to TFLite with onnx2tf
import onnx2tf
import numpy as np
from glob import glob
from PIL import Image

tmp_file = "tmp_tflite_int8_calibration_images.npy"
print(f"Using custom int8 calibration dataset for onnx2tf from {tmp_file}")
data = glob("data/blueberry_cls_v4/train/images/*.jpg")[:400]
images = []
for img in data:
    images.append(np.asarray(Image.open(img).convert('RGB').resize((160, 160))))
images = np.stack(images, 0)
np.save(str(tmp_file), images.astype(np.float32))
np_data = [["input1", tmp_file, [[[[0., 0., 0.]]]], [[[[255., 255., 255.]]]]]]
            
keras_model = onnx2tf.convert(
    input_onnx_file_path="onnx/v4.1/yolov12_160_sm.onnx",
    output_folder_path="onnx/v4.1/save_model/",
    not_use_onnxsim=True,
    verbosity="error",  # note INT8-FP16 activation bug https://github.com/ultralytics/ultralytics/issues/15873
    output_integer_quantized_tflite=True,
    quant_type="per-channel",  # "per-tensor" (faster) or "per-channel" (slower but more accurate)
    custom_input_op_name_np_data_path=np_data,
    disable_group_convolution=True,  # for end-to-end model compatibility
    enable_batchmatmul_unfold=True,  # for end-to-end model compatibility
)

In [ ]:
import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path="onnx/save_model/yolov12_160_sm_full_integer_quant.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("Input details:")
for k, v in input_details[0].items():
    print(f"  {k}: {v}")
print("Output details:")
print(len(output_details))
for output_detail in output_details:
    for k, v in output_detail.items():
        print(f"  {k}: {v}")


### 测试 onnx

In [ ]:
import onnx
import random
import numpy as np
import onnxruntime as ort
import glob

from utils import sigmoid, xywh2xyxy, nms
from PIL import Image
from matplotlib import pyplot as plt

session = ort.InferenceSession("onnx/model_epoch_600_saved_model/model_epoch_600.onnx")
inputs = session.get_inputs()
print("Input name:", inputs[0].name)
print("Input shape:", inputs[0].shape)
print("Input type:", inputs[0].type)

# path = "strawberry_cls/test/images/20250831_101627_jpg.rf.baa9958c0e183de4acc059f56bd3e0d3.jpg"
imgs = glob.glob("strawberry_cls/test/images/*.jpg")
path = imgs[random.randint(0, len(imgs) - 1)]
image = np.asarray(Image.open(path).convert("RGB")).copy()
image = image.astype(np.float32).transpose(2, 0, 1) / 255.0
image = np.expand_dims(image, axis=0)

outputs = session.run(None, {inputs[0].name: image})
anchors = outputs[0][0].transpose(1, 0)

# split
boxes, cls = anchors[:, :4], anchors[:, 4:]
boxes = xywh2xyxy(boxes)
conf = cls.max(axis=1)
cls_id = cls.argmax(axis=1)

# filter
mask = conf > 0.25
boxes = boxes[mask]
cls = cls[mask]
conf = conf[mask]
cls_id = cls_id[mask]

# nms
pred_idx = nms(boxes, conf, iou_threshold=0.45)



# cls2label = {0: "fullripe", 1: "semiripe", 2: "unripe"}
# plt.imshow(image[0].transpose(1, 2, 0))
# for i in pred_idx:
#     x_min, y_min, x_max, y_max = boxes[i]
#     plt.gca().add_patch(plt.Rectangle((x_min, y_min), x_max - x_min, y_max - y_min,
#                                       edgecolor='red', facecolor='none', linewidth=2))
#     plt.text(x_min, y_min - 10, f"{cls2label[cls_id[i]]} {conf[i]:.2f}", color='red', fontsize=12)

In [ ]:
# python: 检查 onnx 输入和值信息并找出含 3200 的维度
import onnx
m = onnx.load("onnx/yolov12_simplified.onnx")
onnx.checker.check_model(m)
def dims_of(vi):
    dims=[]
    t=vi.type.tensor_type
    for d in t.shape.dim:
        if d.HasField("dim_value"):
            dims.append(d.dim_value)
        elif d.HasField("dim_param"):
            dims.append(d.dim_param)
        else:
            dims.append(None)
    return dims

for vi in list(m.graph.input)+list(m.graph.value_info)+list(m.graph.output):
    try:
        ds = dims_of(vi)
    except Exception:
        continue
    if 3200 in ds:
        print("Found 3200 in:", vi.name, ds)

print("Model inputs:")
for i in m.graph.input:
    print(i.name, dims_of(i))

### 量化数据

In [ ]:
import os
import glob
import numpy as np
from PIL import Image

image_paths = sorted(glob.glob("strawberry_cls/train/images/*.jpg"))
if not image_paths:
    raise RuntimeError("No images found in strawberry_cls/train/images")

images = []
for image_path in image_paths:
    image = Image.open(image_path).convert("RGB").resize((640, 640))
    image_array = np.asarray(image, dtype=np.float32)  # 0~255
    images.append(image_array)

train_images = np.stack(images, axis=0).astype(np.float32)  # NHWC
os.makedirs("onnx", exist_ok=True)
np.save("onnx/train_images.npy", train_images)

print("saved:", "onnx/train_images.npy")
print("shape:", train_images.shape)
print("dtype:", train_images.dtype)
print("min/max:", train_images.min(), train_images.max())

### 分析 TFLite int8 量化模型的 sigmoid 查找表

In [2]:
import tensorflow as tf
import numpy as np

model_path = "onnx/v4.1/save_model/yolov12_160_sm_full_integer_quant.tflite"
interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()

# 1. 查看输入输出量化参数
print("=" * 60)
print("输入量化参数:")
for inp in interpreter.get_input_details():
    print(f"  name: {inp['name']}")
    print(f"  shape: {inp['shape']}")
    print(f"  dtype: {inp['dtype']}")
    print(f"  quant: scale={inp['quantization_parameters']['scales']}, "
          f"zp={inp['quantization_parameters']['zero_points']}")

print("\n输出量化参数:")
for out in interpreter.get_output_details():
    print(f"  name: {out['name']}")
    print(f"  shape: {out['shape']}")
    print(f"  dtype: {out['dtype']}")
    print(f"  quant: scale={out['quantization_parameters']['scales']}, "
          f"zp={out['quantization_parameters']['zero_points']}")

# 2. 尝试提取 sigmoid 查找表
# TFLite 的 BuiltinOperator 定义中，SIGMOID 在 int8 模式下会用表实现
# 可以通过解析 flatbuffer 来提取
print("\n" + "=" * 60)
print("Tensor 详细信息:")
for i, t in enumerate(interpreter.get_tensor_details()):
    if 'sigmoid' in t['name'].lower() or 'Sigmoid' in t['name']:
        print(f"  [{i}] {t['name']}: shape={t['shape']}, dtype={t['dtype']}, "
              f"quant={t['quantization_parameters']}")

# 3. 实际推理，观察置信度输出的分布
print("\n" + "=" * 60)
print("实际推理 - 观察置信度输出分布:")

# 用模拟输入跑一次推理
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
dummy_input = np.random.randn(*input_details[0]['shape']).astype(np.float32)
# 量化为 int8
# quantization 返回 (scale, zero_point) 的 tuple
i_scale, i_zp = input_details[0]['quantization']
dummy_input_quant = np.round(dummy_input / i_scale + i_zp).astype(np.int8)
dummy_input_quant = np.clip(dummy_input_quant, -128, 127).astype(np.int8)

interpreter.set_tensor(input_details[0]['index'], dummy_input_quant)
interpreter.invoke()

# 检查所有输出
for out in output_details:
    out_tensor = interpreter.get_tensor(out['index'])
    print(f"\n  Output: {out['name']}, shape={out_tensor.shape}, dtype={out_tensor.dtype}")
    
    # 反量化
    o_scale, o_zp = out['quantization']
    dequantized = (out_tensor.astype(np.float32) - o_zp) * o_scale
    
    # 统计
    unique_vals = np.unique(np.round(dequantized, 5))
    print(f"  反量化后 unique 值数量: {len(unique_vals)}")
    if len(unique_vals) > 0 and len(unique_vals) < 50:
        print(f"  unique 值: {unique_vals}")
    print(f"  min: {dequantized.min():.6f}, max: {dequantized.max():.6f}")
    
    if len(unique_vals) > 1:
        diffs = np.diff(np.sort(unique_vals))
        print(f"  最小步长: {diffs.min():.6f}")
        print(f"  平均步长: {diffs.mean():.6f}")
        print(f"  最大步长: {diffs.max():.6f}")

输入量化参数:
  name: serving_default_input1:0
  shape: [  1 160 160   3]
  dtype: <class 'numpy.int8'>
  quant: scale=[0.00392157], zp=[-128]

输出量化参数:
  name: PartitionedCall:1
  shape: [  1   1 500]
  dtype: <class 'numpy.int8'>
  quant: scale=[0.00390625], zp=[-128]
  name: PartitionedCall:0
  shape: [  1   4 500]
  dtype: <class 'numpy.int8'>
  quant: scale=[1.5087615], zp=[-112]

Tensor 详细信息:
  [190] model_4/tf.math.sigmoid/Sigmoid: shape=[ 1 80 80 16], dtype=<class 'numpy.int8'>, quant={'scales': array([0.00390625], dtype=float32), 'zero_points': array([-128], dtype=int32), 'quantized_dimension': 0}
  [199] model_4/tf.math.sigmoid_1/Sigmoid: shape=[ 1 40 40 32], dtype=<class 'numpy.int8'>, quant={'scales': array([0.00390625], dtype=float32), 'zero_points': array([-128], dtype=int32), 'quantized_dimension': 0}
  [202] model_4/tf.math.sigmoid_2/Sigmoid: shape=[ 1 40 40 32], dtype=<class 'numpy.int8'>, quant={'scales': array([0.00390625], dtype=float32), 'zero_points': array([-128], dtype

In [ ]:
# 提取 TFLite int8 量化模型中的 sigmoid 查找表 (LUT)
# 在 TFLite int8 中, sigmoid/tanh 通过查表实现 (256-entry LUT)
import tensorflow as tf
import numpy as np

model_path = "onnx/v4.1/save_model/yolov12_160_sm_full_integer_quant.tflite"

# 方法1: 用 tf.lite.experimental 的 flatbuffer 解析
try:
    # 读取 flatbuffer 二进制
    with open(model_path, "rb") as f:
        buf = f.read()
    
    # 用 TFLite 的 flatbuffer schema 解析
    model = tf.lite.experimental.make_model_from_flatbuffer(buf)
    
    print("=== 算子列表 ===")
    for i, op in enumerate(model.operator_codes):
        print(f"  [{i}] {op.code} (builtin_code={op.builtin_code})")
    
    print(f"\n子图数量: {len(model.subgraphs)}")
    
    for sg_idx, sg in enumerate(model.subgraphs):
        print(f"\n--- 子图 {sg_idx} ---")
        print(f"  tensors: {len(sg.tensors)}, operators: {len(sg.operators)}, inputs: {sg.inputs}, outputs: {sg.outputs}")
        
        for op_idx, op in enumerate(sg.operators):
            op_code = model.operator_codes[op.opcode_index]
            code_str = str(op_code.builtin_code)
            # 检查是否是 sigmoid
            if 'SIGMOID' in code_str or 'sigmoid' in code_str.lower() or op_code.builtin_code == 18:  # 18 = SIGMOID
                print(f"\n  [算子 {op_idx}] Sigmoid! opcode_index={op.opcode_index}")
                print(f"    inputs: {op.inputs}, outputs: {op.outputs}")
                print(f"    builtin_options: {op.builtin_options}")
                
                # 输入和输出的 tensor
                for t_idx in op.inputs:
                    t = sg.tensors[t_idx]
                    print(f"    input tensor[{t_idx}]: name={t.name!r}, shape={t.shape}, q={(t.quantization.scale, t.quantization.zero_point)}")
                for t_idx in op.outputs:
                    t = sg.tensors[t_idx]
                    print(f"    output tensor[{t_idx}]: name={t.name!r}, shape={t.shape}, q={(t.quantization.scale, t.quantization.zero_point)}")

                # 尝试获取 LUT buffer (可能以 buffer 形式存储)
                if hasattr(op, 'custom_options') and op.custom_options:
                    print(f"    custom_options 长度: {len(op.custom_options)} bytes")
                    if len(op.custom_options) == 256:
                        lut = np.frombuffer(op.custom_options, dtype=np.uint8)
                        print(f"    找到 256-entry LUT!")
                        print(f"    LUT[:16]: {lut[:16]}")
except Exception as e:
    print(f"flatbuffers 解析失败: {e}")
    print("尝试方法2...")

# 方法2: 直接解析 flatbuffer 二进制查找 sigmoid LUT
# 对 flatbuffer 二进制做 naive 搜索，找 256 字节的 LUT 模式
# TFLite 的 sigmoid LUT 是预先计算的: LUT[i] = round(255 * sigmoid((i-128) / 64))
# 理论上的 LUT
def theoretical_sigmoid_lut():
    lut = np.zeros(256, dtype=np.uint8)
    for i in range(256):
        x = (i - 128) / 64.0
        s = 1.0 / (1.0 + np.exp(-x))
        lut[i] = np.clip(np.round(s * 255.0), 0, 255).astype(np.uint8)
    return lut

theoretical_lut = theoretical_sigmoid_lut()
print("\n=== 理论 sigmoid LUT (int8 量化) ===")
print(f"LUT: {theoretical_lut}")
print(f"  LUT[0] (x=-2.0): {theoretical_lut[0]}")
print(f"  LUT[128] (x=0): {theoretical_lut[128]}")
print(f"  LUT[255] (x~1.98): {theoretical_lut[255]}")

# 计算在 TFLite int8 量化下，sigmoid 输出经过反量化后的最小步长
# sigmoid 输出量化为 uint8 [0, 255]，对应浮点值 [0.0, 1.0]
# 输出的 scale 和 zp 决定了最终的浮点分辨率
print("\n=== 分析量化精度 ===")
print("sigmoid 输出量化为 uint8: val_fp = (val_uint8 - zp) * scale")
print(f"理论 LUT 最小非零值: {theoretical_lut[theoretical_lut > 0].min()}")
print(f"理论 LUT 最大值: {theoretical_lut.max()}")

# TFLite sigmoid 通常输出 scale=1/255, zp=0 (即直接映射 uint8 0-255 到 0.0-1.0)
# 但如果有额外的量化参数，精度会不同
scale_default = 1.0 / 255.0
zp_default = 0
print(f"\n默认 sigmoid 量化 (scale=1/255, zp=0):")
print(f"  步长 = {scale_default:.6f}")
lut_fp_default = (theoretical_lut.astype(np.float32) - zp_default) * scale_default
diffs_default = np.diff(np.sort(np.unique(np.round(lut_fp_default, 6))))
print(f"  有效浮点值数量: {len(np.unique(np.round(lut_fp_default, 6)))}")
print(f"  最小步长: {diffs_default.min():.6f}")
print(f"  平均步长: {diffs_default.mean():.6f}")

# 如果输出层有额外的量化 (额外的 scale/zp)
# 从之前的输出我们知道输出的量化参数
# 让我们用实际的 scale/zp 来计算
interpreter = tf.lite.Interpreter(model_path=model_path)
interpreter.allocate_tensors()
output_details = interpreter.get_output_details()
print(f"\n实际输出量化参数 ({len(output_details)} 个输出):")
for out in output_details:
    o_scales = out['quantization_parameters']['scales']
    o_zps = out['quantization_parameters']['zero_points']
    print(f"  {out['name']}: scales={o_scales}, zps={o_zps}")
    if len(o_scales) > 0:
        o_scale = o_scales[0]
        print(f"    该输出层的浮点步长 = {o_scale:.6f}")
        print(f"    对应 sigmoid 的最小可分辨概率变化 ≈ {o_scale:.6f}")

# 如果输出 scale 大约 0.13，说明这不是 sigmoid 的直接输出
# 而是 sigmoid 输出后又经过了一层粗糙的量化
# 我们来验证一下
print("\n=== 分析置信度输出的量化 ===")
print("如果输出反量化后步长 ~0.13，说明:")
print("  1) sigmoid 本身通常是 scale=1/255 ≈ 0.0039 的精度")
print("  2) 0.13 的步长意味着后续层 (可能是 concat 或卷积) 引入了额外的粗量化")
print("  3) 这是多层量化传播导致的精度损失")

# 验证: 如果 sigmoid 后续接了 Conv/Concat，输出的量化步长会变粗
# 找到所有包含 sigmoid 的路径上的 tensor
print("\n=== 追踪 sigmoid 路径上的所有 tensor 量化参数 ===")
for t in interpreter.get_tensor_details():
    name_lower = t['name'].lower()
    if 'sigmoid' in name_lower or 'mul' in name_lower or 'conv' in name_lower:
        q = t['quantization_parameters']
        scale_arr = q['scales']
        print(f"  [{t['index']:3d}] {t['name'][:60]:60s} shape={str(t['shape']):20s} "
              f"scale={scale_arr[0] if len(scale_arr) > 0 else 'N/A':.6f} "
              f"zp={q['zero_points'][0] if len(q['zero_points']) > 0 else 'N/A'}")


2026-07-25 16:08:38.467092: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-25 16:08:39.770769: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784966920.159262   53097 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784966920.251680   53097 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784966921.392798   53097 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

flatbuffers 解析失败: module 'tensorflow._api.v2.lite.experimental' has no attribute 'make_model_from_flatbuffer'
尝试方法2...

=== 理论 sigmoid LUT (int8 量化) ===
LUT: [ 30  31  31  32  32  33  33  33  34  34  35  35  36  36  37  37  38  38
  39  39  40  40  41  41  42  43  43  44  44  45  45  46  47  47  48  48
  49  50  50  51  51  52  53  53  54  55  55  56  57  57  58  59  60  60
  61  62  62  63  64  65  65  66  67  68  69  69  70  71  72  73  73  74
  75  76  77  78  78  79  80  81  82  83  84  84  85  86  87  88  89  90
  91  92  93  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142
 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160
 161 162 162 163 164 165 166 167 168 169 170 171 171 172 173 174 175 176
 177 177 178 179 180 181 182 182 183 184 185 186 186 187 188 189 190 190
 191 192 193 193 194 195 195 196 197 19

/root/autodl-tmp/envs/yolo/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


TypeError: unsupported format string passed to numpy.ndarray.__format__